In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

if not os.path.exists("scripts/data.py"):
    raise FileNotFoundError(
        "Run this notebook from the repository root or the scripts directory."
    )

if "scripts" not in sys.path:
    sys.path.insert(0, "scripts")

os.makedirs("results", exist_ok=True)
os.makedirs("cache", exist_ok=True)

In [ ]:

# Config

# Run validation first, then test.
RUN_MODE = "test"  # "validation" or "test"

VALID_RUN_MODES = {"validation", "test"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE}")

# Choose any subset from "text", "progression", and "header".
COMPONENTS = ["text", "progression"]
# COMPONENTS = ["text"]
# COMPONENTS = ["text", "progression", "header"]
K = 5
ENCODER_SHORT_NAME = "pubmedbert"
TEXT_FUSION_MODE = "late_prob_timeline"

MASTER_PATH = "data/PMCOA_samples_01234_overlap_clean_split_headers.tsv"

LABEL_ORDER = ["I", "M", "R", "D", "NoIMRaD"]

# Layout-prior settings
PRIOR_ALPHA = 1.0

# Used only in validation mode
PRIOR_WEIGHT_GRID = [
    0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0
]

# In test mode, leave this as None to load the value selected on validation
# Set a numeric value only as an explicit manual override
SELECTED_PRIOR_WEIGHT = None

# Test-mode document-level confidence intervals
RUN_BOOTSTRAP = True
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95


component_name = "_plus_".join(COMPONENTS)

if "text" in COMPONENTS:
    META_RUN_NAME = (
        f"{component_name}_{ENCODER_SHORT_NAME}_k{K}_"
        f"{TEXT_FUSION_MODE}_logreg"
    )
else:
    META_RUN_NAME = f"{component_name}_k{K}_logreg"

if RUN_MODE == "validation":
    LOCAL_META_ROOT = "results/validation/local_meta"
    GLOBAL_OUTPUT_ROOT = "results/validation/global_layout_prior"
    EVAL_SPLIT_NAME = "validation"
    PRIOR_SOURCE_NAME = "train"

else:
    LOCAL_META_ROOT = "results/test/local_meta"
    GLOBAL_OUTPUT_ROOT = "results/test/global_layout_prior"
    EVAL_SPLIT_NAME = "test"
    PRIOR_SOURCE_NAME = "train+validation"

META_PRED_PATH = os.path.join(
    LOCAL_META_ROOT,
    META_RUN_NAME,
    "predictions.csv",
)

OUTPUT_DIR = os.path.join(
    GLOBAL_OUTPUT_ROOT,
    META_RUN_NAME,
)

VALIDATION_SELECTION_DIR = os.path.join(
    "results",
    "validation",
    "global_layout_prior",
    META_RUN_NAME,
)

SELECTED_WEIGHT_PATH = os.path.join(
    VALIDATION_SELECTION_DIR,
    "selected_prior_weight.json",
)

print("Run mode:", RUN_MODE)
print("Components:", COMPONENTS)
print("Meta run name:", META_RUN_NAME)
print("Evaluation split:", EVAL_SPLIT_NAME)
print("Prior source:", PRIOR_SOURCE_NAME)
print("Local prediction path:", META_PRED_PATH)
print("Output directory:", OUTPUT_DIR)

if RUN_MODE == "test":
    print("Validation-selected weight path:", SELECTED_WEIGHT_PATH)


In [ ]:
# load data and construct document prior

from data import (
    DatasetConfig,
    load_blocks,
    split_blocks,
    make_layout_table,
)


def assign_canonical_section_order(df):
    out = df.copy()

    required = {"paper_id", "section_id", "start_progression"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(
            "Cannot construct canonical section order. Missing columns: "
            f"{sorted(missing)}"
        )

    if out[["paper_id", "section_id", "start_progression"]].isna().any().any():
        raise ValueError(
            "Missing paper_id, section_id, or start_progression values were "
            "found; canonical ordering cannot be assigned safely."
        )

    if "source_section_order" not in out.columns:
        if "section_order" in out.columns:
            out["source_section_order"] = out["section_order"]
        else:
            out["source_section_order"] = np.nan

    sort_cols = ["paper_id", "start_progression"]
    if "end_progression" in out.columns:
        sort_cols.append("end_progression")
    sort_cols.append("section_id")

    out = (
        out.sort_values(sort_cols, kind="mergesort")
        .reset_index(drop=True)
    )

    out["canonical_section_order"] = (
        out.groupby("paper_id", sort=False).cumcount() + 1
    ).astype(int)

    out["section_order"] = out["canonical_section_order"]

    return out


cfg = DatasetConfig(master_path=MASTER_PATH)

blocks_df = load_blocks(
    config=cfg,
    k=K,
    fusion="early",
)

train_df, val_df, test_df = split_blocks(blocks_df)

train_df = assign_canonical_section_order(train_df)
val_df = assign_canonical_section_order(val_df)
test_df = assign_canonical_section_order(test_df)

if RUN_MODE == "validation":
    prior_source_df = train_df.copy()
    eval_reference_df = val_df.copy()
else:
    prior_source_df = pd.concat(
        [train_df, val_df],
        axis=0,
        ignore_index=True,
    )
    prior_source_df = assign_canonical_section_order(prior_source_df)
    eval_reference_df = test_df.copy()

prior_layout_df = make_layout_table(prior_source_df)
eval_layout_reference_df = make_layout_table(eval_reference_df)

print("Canonical order: start_progression -> end_progression -> section_id")
print("Prior-source blocks:", len(prior_source_df))
print("Prior-source articles:", prior_source_df["paper_id"].nunique())
print("Evaluation blocks:", len(eval_reference_df))
print("Evaluation articles:", eval_reference_df["paper_id"].nunique())

In [ ]:

def parse_layout(layout):
    return str(layout).split("->")


def has_consecutive_duplicates(layout):
    labels = parse_layout(layout)
    return any(
        labels[i] == labels[i + 1]
        for i in range(len(labels) - 1)
    )


prior_layout_df = prior_layout_df.copy()
prior_layout_df["has_consecutive_duplicates"] = (
    prior_layout_df["layout"].apply(has_consecutive_duplicates)
)

bad_prior_layouts = prior_layout_df[
    prior_layout_df["has_consecutive_duplicates"]
].copy()

# Only consecutive duplicate layouts are excluded. Non-consecutive repeated
# labels remain eligible because they occur in the current dataset definition.
prior_layout_df_for_prior = prior_layout_df[
    ~prior_layout_df["has_consecutive_duplicates"]
].copy()

print("Prior-source layouts:", len(prior_layout_df))
print(
    "Layouts excluded for consecutive duplicates:",
    len(bad_prior_layouts),
)
print("Layouts retained for prior:", len(prior_layout_df_for_prior))

if len(bad_prior_layouts) > 0:
    display(
        bad_prior_layouts[["paper_id", "layout", "num_blocks"]].head(20)
    )


In [ ]:

def build_layout_prior(layout_df, alpha=1.0):
    if alpha < 0:
        raise ValueError("alpha must be non-negative")

    layout_counts = (
        layout_df["layout"]
        .value_counts()
        .rename_axis("layout")
        .reset_index(name="count")
    )

    layout_counts["num_blocks"] = (
        layout_counts["layout"].apply(lambda x: len(parse_layout(x)))
    )

    total = int(layout_counts["count"].sum())
    num_layouts = int(len(layout_counts))

    layout_counts["prior"] = (
        (layout_counts["count"] + alpha)
        / (total + alpha * num_layouts)
    )
    layout_counts["log_prior"] = np.log(layout_counts["prior"])

    # Stable order gives deterministic tie-breaking.
    layout_counts = layout_counts.sort_values(
        ["num_blocks", "count", "layout"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

    return layout_counts


layout_prior_df = build_layout_prior(
    prior_layout_df_for_prior,
    alpha=PRIOR_ALPHA,
)

print("Number of observed layout templates:", len(layout_prior_df))
print("Prior-source papers used:", prior_layout_df_for_prior["paper_id"].nunique())
print()
display(layout_prior_df.head(20))


In [ ]:
# load

if not os.path.exists(META_PRED_PATH):
    raise FileNotFoundError(
        f"Missing local prediction file: {META_PRED_PATH}\n"
        f"Run 02_metalearner.ipynb with RUN_MODE='{RUN_MODE}' and the matching configuration first."
    )

meta_eval_df = pd.read_csv(META_PRED_PATH)

required_cols = {
    "paper_id",
    "section_id",
    "y_true",
    "y_pred",
    *[f"meta_prob_{label}" for label in LABEL_ORDER],
}
missing_cols = required_cols - set(meta_eval_df.columns)
if missing_cols:
    raise ValueError(
        f"Local prediction file is missing columns: {sorted(missing_cols)}"
    )

if meta_eval_df.duplicated(["paper_id", "section_id"]).any():
    raise ValueError("Duplicate paper_id + section_id rows were found.")

if meta_eval_df[list(required_cols)].isna().any().any():
    raise ValueError("Missing values were found in required prediction columns.")

prob_cols = [f"meta_prob_{label}" for label in LABEL_ORDER]
prob_sums = meta_eval_df[prob_cols].sum(axis=1)
if not np.allclose(prob_sums.values, 1.0, atol=1e-5):
    raise ValueError("Local meta probabilities do not sum to 1.")

prediction_papers = set(meta_eval_df["paper_id"].astype(str))
expected_eval_papers = set(eval_reference_df["paper_id"].astype(str))
prior_papers = set(prior_source_df["paper_id"].astype(str))

if prediction_papers != expected_eval_papers:
    missing_predictions = expected_eval_papers - prediction_papers
    unexpected_predictions = prediction_papers - expected_eval_papers
    raise ValueError(
        "Prediction/evaluation paper mismatch. "
        f"Missing papers: {len(missing_predictions)}; "
        f"unexpected papers: {len(unexpected_predictions)}"
    )

if prediction_papers & prior_papers:
    raise ValueError("Prior-source and evaluation articles overlap.")

if len(meta_eval_df) != len(eval_reference_df):
    raise ValueError(
        "Prediction/evaluation block-count mismatch: "
        f"{len(meta_eval_df)} vs. {len(eval_reference_df)}"
    )

print("Loaded local predictions:", meta_eval_df.shape)
print("Evaluation articles:", meta_eval_df["paper_id"].nunique())
print("Evaluation blocks:", len(meta_eval_df))
print("Duplicate section rows: 0")
print("Prior/evaluation article overlap: 0")


In [ ]:
def attach_reference_order(meta_df, reference_df):
    pred = meta_df.copy()
    ref = reference_df.copy()

    if "source_section_order" not in pred.columns:
        if "section_order" in pred.columns:
            pred["source_section_order"] = pred["section_order"]
        else:
            pred["source_section_order"] = np.nan

    pred["_paper_key"] = pred["paper_id"].astype(str)
    pred["_section_key"] = pred["section_id"].astype(str)
    ref["_paper_key"] = ref["paper_id"].astype(str)
    ref["_section_key"] = ref["section_id"].astype(str)

    reference_cols = [
        "_paper_key",
        "_section_key",
        "canonical_section_order",
    ]

    if ref.duplicated(["_paper_key", "_section_key"]).any():
        raise ValueError(
            "Duplicate paper_id + section_id keys were found in the "
            "evaluation reference data."
        )

    pred = pred.merge(
        ref[reference_cols],
        on=["_paper_key", "_section_key"],
        how="left",
        validate="one_to_one",
    )

    if pred["canonical_section_order"].isna().any():
        missing_rows = pred.loc[
            pred["canonical_section_order"].isna(),
            ["paper_id", "section_id"],
        ]
        raise ValueError(
            "Some prediction rows could not be matched to the canonical "
            "evaluation reference order. Example rows:\n"
            f"{missing_rows.head(10).to_string(index=False)}"
        )

    pred["canonical_section_order"] = (
        pred["canonical_section_order"].astype(int)
    )
    pred["section_order"] = pred["canonical_section_order"]

    pred = pred.drop(columns=["_paper_key", "_section_key"])
    return pred


def get_sort_columns(df):
    candidate_cols = [
        "paper_id",
        "canonical_section_order",
        "section_id",
    ]
    sort_cols = [col for col in candidate_cols if col in df.columns]

    if "paper_id" not in sort_cols:
        raise ValueError("paper_id is required for sorting")
    if "canonical_section_order" not in sort_cols:
        raise ValueError("canonical_section_order is required for sorting")

    return sort_cols


def sort_prediction_rows(df):
    return (
        df.copy()
        .sort_values(get_sort_columns(df), kind="mergesort")
        .reset_index(drop=True)
    )


meta_eval_df = attach_reference_order(meta_eval_df, eval_reference_df)
meta_eval_df = sort_prediction_rows(meta_eval_df)

source_numeric = pd.to_numeric(
    meta_eval_df["source_section_order"],
    errors="coerce",
)
order_changed_mask = (
    source_numeric.notna()
    & (source_numeric != meta_eval_df["canonical_section_order"])
)
ordering_corrections_df = meta_eval_df.loc[
    order_changed_mask,
    [
        col for col in [
            "paper_id",
            "pmcid",
            "section_id",
            "section_header",
            "start_progression",
            "end_progression",
            "source_section_order",
            "canonical_section_order",
        ]
        if col in meta_eval_df.columns
    ],
].copy()

print("Sort columns:", get_sort_columns(meta_eval_df))
print("Rows whose source order was corrected:", len(ordering_corrections_df))
print(
    "Articles whose source order was corrected:",
    ordering_corrections_df["paper_id"].nunique(),
)
if len(ordering_corrections_df) > 0:
    display(ordering_corrections_df.head(20))

display(meta_eval_df.head())

In [ ]:

# Gloabl decoding function

def build_candidate_lookup(layout_prior_df, label_order):
    label_to_index = {label: i for i, label in enumerate(label_order)}
    lookup = {}

    for _, row in layout_prior_df.iterrows():
        labels = parse_layout(row["layout"])

        unknown_labels = set(labels) - set(label_order)
        if unknown_labels:
            raise ValueError(
                f"Unknown labels in layout {row['layout']}: {unknown_labels}"
            )

        candidate = {
            "layout": row["layout"],
            "labels": labels,
            "label_indices": np.array(
                [label_to_index[label] for label in labels],
                dtype=int,
            ),
            "log_prior": float(row["log_prior"]),
            "count": int(row["count"]),
        }
        lookup.setdefault(len(labels), []).append(candidate)

    return lookup


candidate_lookup = build_candidate_lookup(layout_prior_df, LABEL_ORDER)

print("Candidate template counts by block length:")
for n in sorted(candidate_lookup):
    print(f"  {n} blocks: {len(candidate_lookup[n])}")


In [ ]:
def decode_one_paper(
    group,
    candidate_lookup,
    label_order,
    prior_weight,
    eps=1e-12,
):
    group = sort_prediction_rows(group)
    n = len(group)
    candidates = candidate_lookup.get(n, [])

    out = group.copy()
    local_layout = "->".join(out["y_pred"].astype(str))
    out["local_layout"] = local_layout

    if len(candidates) == 0:
        out["global_y_pred"] = out["y_pred"].values
        out["global_layout"] = local_layout
        out["global_local_log_likelihood"] = np.nan
        out["global_layout_log_prior"] = np.nan
        out["global_layout_prior_count"] = np.nan
        out["global_score"] = np.nan
        out["global_used_fallback"] = True
        out["global_changed"] = False
        return out

    probability_matrix = out[
        [f"meta_prob_{label}" for label in label_order]
    ].to_numpy(dtype=float)
    probability_matrix = np.clip(probability_matrix, eps, 1.0)

    best_candidate = None
    best_score = -np.inf
    best_local_log_likelihood = np.nan

    row_indices = np.arange(n)

    for candidate in candidates:
        local_log_likelihood = float(
            np.log(
                probability_matrix[
                    row_indices,
                    candidate["label_indices"],
                ]
            ).sum()
        )
        score = (
            local_log_likelihood
            + prior_weight * candidate["log_prior"]
        )

        if score > best_score:
            best_score = score
            best_candidate = candidate
            best_local_log_likelihood = local_log_likelihood

    out["global_y_pred"] = best_candidate["labels"]
    out["global_layout"] = best_candidate["layout"]
    out["global_local_log_likelihood"] = best_local_log_likelihood
    out["global_layout_log_prior"] = best_candidate["log_prior"]
    out["global_layout_prior_count"] = best_candidate["count"]
    out["global_score"] = best_score
    out["global_used_fallback"] = False
    out["global_changed"] = (
        out["global_y_pred"].astype(str)
        != out["y_pred"].astype(str)
    )

    return out


def decode_all_papers(
    meta_df,
    candidate_lookup,
    label_order,
    prior_weight,
):
    decoded_groups = []

    for _, group in meta_df.groupby("paper_id", sort=False):
        decoded_groups.append(
            decode_one_paper(
                group=group,
                candidate_lookup=candidate_lookup,
                label_order=label_order,
                prior_weight=prior_weight,
            )
        )

    return pd.concat(decoded_groups, axis=0, ignore_index=True)


In [ ]:
# evaluate and prior weight selction

from evaluate import (
    evaluate_predictions,
    print_evaluation_summary,
    evaluate_layout_predictions,
    print_layout_evaluation_summary,
)

ORDER_COLS = [
    col
    for col in ["canonical_section_order", "section_id"]
    if col in meta_eval_df.columns
]


def evaluate_decoded_predictions(decoded_df):
    local_section = evaluate_predictions(
        y_true=decoded_df["y_true"].values,
        y_pred=decoded_df["y_pred"].values,
        y_prob=None,
        label_order=LABEL_ORDER,
    )
    global_section = evaluate_predictions(
        y_true=decoded_df["y_true"].values,
        y_pred=decoded_df["global_y_pred"].values,
        y_prob=None,
        label_order=LABEL_ORDER,
    )

    local_layout_metrics, local_layout_df = evaluate_layout_predictions(
        df=decoded_df,
        paper_id_col="paper_id",
        true_col="y_true",
        pred_col="y_pred",
        order_cols=ORDER_COLS,
    )
    global_layout_metrics, global_layout_df = evaluate_layout_predictions(
        df=decoded_df,
        paper_id_col="paper_id",
        true_col="y_true",
        pred_col="global_y_pred",
        order_cols=ORDER_COLS,
    )

    return {
        "local_section_metrics": local_section[0],
        "local_section_report": local_section[1],
        "local_section_confusion": local_section[2],
        "global_section_metrics": global_section[0],
        "global_section_report": global_section[1],
        "global_section_confusion": global_section[2],
        "local_layout_metrics": local_layout_metrics,
        "local_layout_df": local_layout_df,
        "global_layout_metrics": global_layout_metrics,
        "global_layout_df": global_layout_df,
    }


def summarize_lambda(prior_weight):
    decoded_df = decode_all_papers(
        meta_df=meta_eval_df,
        candidate_lookup=candidate_lookup,
        label_order=LABEL_ORDER,
        prior_weight=prior_weight,
    )
    results = evaluate_decoded_predictions(decoded_df)

    fallback_articles = int(
        decoded_df.groupby("paper_id")["global_used_fallback"].first().sum()
    )

    return {
        "prior_weight": float(prior_weight),
        "section_accuracy": results["global_section_metrics"]["accuracy"],
        "section_macro_f1": results["global_section_metrics"]["macro_f1"],
        "layout_exact_match_accuracy": results["global_layout_metrics"][
            "layout_exact_match_accuracy"
        ],
        "mean_layout_edit_distance": results["global_layout_metrics"][
            "mean_layout_edit_distance"
        ],
        "mean_layout_normalized_edit_distance": results[
            "global_layout_metrics"
        ]["mean_layout_normalized_edit_distance"],
        "changed_blocks": int(decoded_df["global_changed"].sum()),
        "fallback_articles": fallback_articles,
        "fallback_rate": (
            fallback_articles / decoded_df["paper_id"].nunique()
        ),
    }


In [ ]:

if RUN_MODE == "validation":
    lambda_rows = []

    for weight in PRIOR_WEIGHT_GRID:
        print(f"Evaluating prior_weight={weight} ...")
        lambda_rows.append(summarize_lambda(weight))

    lambda_search_df = pd.DataFrame(lambda_rows)

    # Primary criterion: higher exact-layout accuracy.
    # Tie-breakers: lower normalized edit distance, then lower lambda.
    ranked_lambda_df = lambda_search_df.sort_values(
        [
            "layout_exact_match_accuracy",
            "mean_layout_normalized_edit_distance",
            "prior_weight",
        ],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    selected_prior_weight = float(ranked_lambda_df.iloc[0]["prior_weight"])

    print()
    print("Validation lambda search:")
    display(lambda_search_df)
    print("Selected prior weight:", selected_prior_weight)

else:
    lambda_search_df = None

    if SELECTED_PRIOR_WEIGHT is not None:
        selected_prior_weight = float(SELECTED_PRIOR_WEIGHT)
        print(
            "Using explicit prior-weight override:",
            selected_prior_weight,
        )
    else:
        if not os.path.exists(SELECTED_WEIGHT_PATH):
            raise FileNotFoundError(
                "Validation-selected prior weight was not found: "
                f"{SELECTED_WEIGHT_PATH}\n"
                "Run this notebook in validation mode first."
            )

        with open(SELECTED_WEIGHT_PATH, "r", encoding="utf-8") as f:
            selected_weight_record = json.load(f)

        selected_prior_weight = float(
            selected_weight_record["selected_prior_weight"]
        )
        print(
            "Loaded validation-selected prior weight:",
            selected_prior_weight,
        )


decoded_eval_df = decode_all_papers(
    meta_df=meta_eval_df,
    candidate_lookup=candidate_lookup,
    label_order=LABEL_ORDER,
    prior_weight=selected_prior_weight,
)

final_results = evaluate_decoded_predictions(decoded_eval_df)

print("Decoded split:", EVAL_SPLIT_NAME)
print("Selected prior weight:", selected_prior_weight)
print("Decoded rows:", len(decoded_eval_df))
print("Decoded articles:", decoded_eval_df["paper_id"].nunique())


In [ ]:
# local vs. global results

print("LOCAL SECTION-LEVEL PERFORMANCE")
print_evaluation_summary(
    final_results["local_section_metrics"],
    final_results["local_section_confusion"],
)

print()
print("GLOBAL-DECODED SECTION-LEVEL PERFORMANCE")
print_evaluation_summary(
    final_results["global_section_metrics"],
    final_results["global_section_confusion"],
)

print()
print("LOCAL DOCUMENT-LEVEL PERFORMANCE")
print_layout_evaluation_summary(
    final_results["local_layout_metrics"]
)

print()
print("GLOBAL-DECODED DOCUMENT-LEVEL PERFORMANCE")
print_layout_evaluation_summary(
    final_results["global_layout_metrics"]
)


In [ ]:

section_comparison_df = pd.DataFrame([
    {
        "model": "local_meta_only",
        "accuracy": final_results["local_section_metrics"]["accuracy"],
        "macro_precision": final_results["local_section_metrics"][
            "macro_precision"
        ],
        "macro_recall": final_results["local_section_metrics"][
            "macro_recall"
        ],
        "macro_f1": final_results["local_section_metrics"]["macro_f1"],
        "num_blocks": final_results["local_section_metrics"]["num_samples"],
    },
    {
        "model": "local_meta_plus_global_prior",
        "accuracy": final_results["global_section_metrics"]["accuracy"],
        "macro_precision": final_results["global_section_metrics"][
            "macro_precision"
        ],
        "macro_recall": final_results["global_section_metrics"][
            "macro_recall"
        ],
        "macro_f1": final_results["global_section_metrics"]["macro_f1"],
        "num_blocks": final_results["global_section_metrics"]["num_samples"],
    },
])

article_comparison_df = pd.DataFrame([
    {
        "model": "local_meta_only",
        "layout_exact_match_accuracy": final_results[
            "local_layout_metrics"
        ]["layout_exact_match_accuracy"],
        "mean_layout_edit_distance": final_results[
            "local_layout_metrics"
        ]["mean_layout_edit_distance"],
        "mean_layout_normalized_edit_distance": final_results[
            "local_layout_metrics"
        ]["mean_layout_normalized_edit_distance"],
        "num_articles": final_results["local_layout_metrics"]["num_articles"],
    },
    {
        "model": "local_meta_plus_global_prior",
        "layout_exact_match_accuracy": final_results[
            "global_layout_metrics"
        ]["layout_exact_match_accuracy"],
        "mean_layout_edit_distance": final_results[
            "global_layout_metrics"
        ]["mean_layout_edit_distance"],
        "mean_layout_normalized_edit_distance": final_results[
            "global_layout_metrics"
        ]["mean_layout_normalized_edit_distance"],
        "num_articles": final_results["global_layout_metrics"]["num_articles"],
    },
])

print("Section-level comparison:")
display(section_comparison_df)

print("Document-level comparison:")
display(article_comparison_df)


In [ ]:

# Diagnostics useful for the later disagreement/error analysis.
local_correct = decoded_eval_df["y_pred"] == decoded_eval_df["y_true"]
global_correct = (
    decoded_eval_df["global_y_pred"] == decoded_eval_df["y_true"]
)
changed = decoded_eval_df["global_changed"]

conditions = [
    (~local_correct) & global_correct,
    local_correct & (~global_correct),
    local_correct & global_correct,
    (~local_correct) & (~global_correct) & changed,
    (~local_correct) & (~global_correct) & (~changed),
]
choices = [
    "global_corrected_local_error",
    "global_harmed_local_correct",
    "correct_before_and_after",
    "changed_but_still_wrong",
    "unchanged_and_wrong",
]

decoded_eval_df["global_change_outcome"] = np.select(
    conditions,
    choices,
    default="other",
)

change_summary_df = (
    decoded_eval_df["global_change_outcome"]
    .value_counts()
    .rename_axis("outcome")
    .reset_index(name="section_blocks")
)
change_summary_df["percentage"] = (
    100 * change_summary_df["section_blocks"] / len(decoded_eval_df)
)

fallback_by_article = (
    decoded_eval_df.groupby("paper_id")["global_used_fallback"]
    .first()
)

print("Section-level decoder outcomes:")
display(change_summary_df)
print(
    "Fallback articles:",
    int(fallback_by_article.sum()),
    f"({100 * fallback_by_article.mean():.2f}%)",
)


In [ ]:
# document-level bootstrap CI (test)

def bootstrap_document_metrics(
    local_layout_df,
    global_layout_df,
    n_bootstrap=5000,
    seed=42,
    ci_level=0.95,
):
    metric_cols = [
        "layout_exact_match",
        "layout_edit_distance",
        "layout_normalized_edit_distance",
    ]

    local_small = local_layout_df[["paper_id", *metric_cols]].copy()
    global_small = global_layout_df[["paper_id", *metric_cols]].copy()

    merged = local_small.merge(
        global_small,
        on="paper_id",
        how="inner",
        suffixes=("_local", "_global"),
        validate="one_to_one",
    )

    if len(merged) != len(local_layout_df) or len(merged) != len(global_layout_df):
        raise ValueError("Local/global article tables do not align.")

    local_exact = merged["layout_exact_match_local"].astype(float).to_numpy()
    global_exact = merged["layout_exact_match_global"].astype(float).to_numpy()
    local_edit = merged["layout_edit_distance_local"].astype(float).to_numpy()
    global_edit = merged["layout_edit_distance_global"].astype(float).to_numpy()
    local_norm = merged[
        "layout_normalized_edit_distance_local"
    ].astype(float).to_numpy()
    global_norm = merged[
        "layout_normalized_edit_distance_global"
    ].astype(float).to_numpy()

    arrays = {
        "local_layout_exact_match_accuracy": local_exact,
        "global_layout_exact_match_accuracy": global_exact,
        "local_mean_layout_edit_distance": local_edit,
        "global_mean_layout_edit_distance": global_edit,
        "local_mean_layout_normalized_edit_distance": local_norm,
        "global_mean_layout_normalized_edit_distance": global_norm,
        # Positive values indicate improvement by the global decoder.
        "exact_match_accuracy_difference_global_minus_local": (
            global_exact - local_exact
        ),
        "normalized_edit_distance_reduction_local_minus_global": (
            local_norm - global_norm
        ),
    }

    rng = np.random.default_rng(seed)
    n_articles = len(merged)
    bootstrap_values = {
        metric: np.empty(n_bootstrap, dtype=float)
        for metric in arrays
    }

    for b in range(n_bootstrap):
        sampled_indices = rng.integers(
            0,
            n_articles,
            size=n_articles,
        )
        for metric, values in arrays.items():
            bootstrap_values[metric][b] = values[sampled_indices].mean()

    alpha = 1.0 - ci_level
    lower_q = 100 * alpha / 2
    upper_q = 100 * (1 - alpha / 2)

    rows = []
    for metric, values in arrays.items():
        ci_lower, ci_upper = np.percentile(
            bootstrap_values[metric],
            [lower_q, upper_q],
        )
        rows.append({
            "metric": metric,
            "estimate": float(values.mean()),
            "ci_lower": float(ci_lower),
            "ci_upper": float(ci_upper),
            "ci_level": ci_level,
            "n_bootstrap": n_bootstrap,
            "bootstrap_unit": "article",
            "num_articles": n_articles,
        })

    return pd.DataFrame(rows)


bootstrap_ci_df = None

if RUN_MODE == "test" and RUN_BOOTSTRAP:
    bootstrap_ci_df = bootstrap_document_metrics(
        local_layout_df=final_results["local_layout_df"],
        global_layout_df=final_results["global_layout_df"],
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED,
        ci_level=CI_LEVEL,
    )
    display(bootstrap_ci_df)
else:
    print("Document-level bootstrap skipped.")


In [ ]:

# Save outputs

def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if np.isscalar(value) else False:
        return None
    return value


os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validation selection artifacts.
if RUN_MODE == "validation":
    lambda_search_df.to_csv(
        os.path.join(OUTPUT_DIR, "lambda_search_results.csv"),
        index=False,
    )

    selected_weight_record = {
        "selected_prior_weight": selected_prior_weight,
        "selection_split": "validation",
        "layout_prior_source": "train",
        "primary_selection_metric": "layout_exact_match_accuracy",
        "tie_breaker": "mean_layout_normalized_edit_distance",
        "components": COMPONENTS,
        "meta_run_name": META_RUN_NAME,
        "k": K,
        "encoder_short_name": ENCODER_SHORT_NAME,
        "text_fusion_mode": TEXT_FUSION_MODE,
    }
    with open(SELECTED_WEIGHT_PATH, "w", encoding="utf-8") as f:
        json.dump(selected_weight_record, f, indent=2)

# Core decoded outputs.
decoded_eval_df.to_csv(
    os.path.join(OUTPUT_DIR, "predictions.csv"),
    index=False,
)
layout_prior_df.to_csv(
    os.path.join(OUTPUT_DIR, "layout_prior.csv"),
    index=False,
)
section_comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "section_level_comparison.csv"),
    index=False,
)
article_comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "article_level_comparison.csv"),
    index=False,
)
change_summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "decoder_change_summary.csv"),
    index=False,
)
ordering_corrections_df.to_csv(
    os.path.join(OUTPUT_DIR, "ordering_corrections.csv"),
    index=False,
)

# Section-level reports and confusion matrices.
final_results["local_section_report"].to_csv(
    os.path.join(OUTPUT_DIR, "section_classification_report_local.csv"),
    index=True,
)
final_results["global_section_report"].to_csv(
    os.path.join(OUTPUT_DIR, "section_classification_report_global.csv"),
    index=True,
)
final_results["local_section_confusion"].to_csv(
    os.path.join(OUTPUT_DIR, "section_confusion_matrix_local.csv"),
    index=True,
)
final_results["global_section_confusion"].to_csv(
    os.path.join(OUTPUT_DIR, "section_confusion_matrix_global.csv"),
    index=True,
)

# Article-level prediction tables.
final_results["local_layout_df"].to_csv(
    os.path.join(OUTPUT_DIR, "article_level_layout_predictions_local.csv"),
    index=False,
)
final_results["global_layout_df"].to_csv(
    os.path.join(OUTPUT_DIR, "article_level_layout_predictions_global.csv"),
    index=False,
)

# Metrics JSON files.
json_outputs = {
    "section_metrics_local.json": final_results["local_section_metrics"],
    "section_metrics_global.json": final_results["global_section_metrics"],
    "article_level_metrics_local.json": final_results["local_layout_metrics"],
    "article_level_metrics_global.json": final_results["global_layout_metrics"],
}
for filename, payload in json_outputs.items():
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        json.dump(to_jsonable(payload), f, indent=2)

if bootstrap_ci_df is not None:
    bootstrap_ci_df.to_csv(
        os.path.join(OUTPUT_DIR, "article_level_bootstrap_ci.csv"),
        index=False,
    )

config = {
    "run_mode": RUN_MODE,
    "model_stage": "global_layout_prior",
    "components": COMPONENTS,
    "component_name": component_name,
    "meta_run_name": META_RUN_NAME,
    "local_meta_prediction_path": META_PRED_PATH,
    "layout_prior_source": PRIOR_SOURCE_NAME,
    "layout_prior_alpha": PRIOR_ALPHA,
    "selected_prior_weight": selected_prior_weight,
    "selected_on_validation": True,
    "k": K,
    "encoder_short_name": ENCODER_SHORT_NAME,
    "text_fusion_mode": TEXT_FUSION_MODE,
    "label_order": LABEL_ORDER,
    "section_ordering": (
        "canonical progression order: start_progression, "
        "end_progression, section_id"
    ),
    "source_section_order_preserved_as": "source_section_order",
    "rows_with_corrected_source_order": len(ordering_corrections_df),
    "articles_with_corrected_source_order": (
        ordering_corrections_df["paper_id"].nunique()
    ),
    "split_evaluated": EVAL_SPLIT_NAME,
    "uses_test_split": RUN_MODE == "test",
    "num_blocks": len(decoded_eval_df),
    "num_articles": decoded_eval_df["paper_id"].nunique(),
    "fallback_articles": int(fallback_by_article.sum()),
    "fallback_rate": float(fallback_by_article.mean()),
    "bootstrap_enabled": bool(RUN_MODE == "test" and RUN_BOOTSTRAP),
    "n_bootstrap": N_BOOTSTRAP if RUN_MODE == "test" and RUN_BOOTSTRAP else None,
    "bootstrap_unit": "article" if RUN_MODE == "test" and RUN_BOOTSTRAP else None,
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(to_jsonable(config), f, indent=2)

print("Saved outputs to:", OUTPUT_DIR)
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" ", filename)
